# Section 0 - Config

In [ ]:
from pathlib import Path
import shutil, subprocess

REPO_URL = "https://github.com/ZeusGav/evaluating-llm-explanations-for-aml.git"
REPO_DIR = Path("/content/ba_pipeline")
PIPE     = REPO_DIR / "ba_pipeline"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

CFG = {
    "seed": 42,
    "pipe": PIPE,
    "llm_runs_dir": PIPE / "llm_outputs" / "runs",
    "shap_dir":     PIPE / "shap_outputs" / "local" / "A",
    "runs": [
        "run_001_benchmark",
        "run_002_stability",
        "run_003_ablation_weaker_model",
        "run_004_constraints_on",
        "run_005_negative_control_mismatched_pairing",
        "run_006_judge_stability",
    ],
    "judge": {
        "model": "gpt-5.2",
        "n_retries": 3,
        "sleep_success_s": 0.05,
        "sleep_retry_base_s": 1.0,
    },
}

print("✅ Repo dir:", REPO_DIR)
print("✅ Pipeline dir:", PIPE)
print("✅ Results dir (existing):", PIPE / "results")

✅ Repo dir: /content/ba_pipeline
✅ Pipeline dir: /content/ba_pipeline/ba_pipeline
✅ Results dir (existing): /content/ba_pipeline/ba_pipeline/results


# Section 1 - Join Data

In [ ]:
# JOIN: SHAP & LLM outputs

import json, random
from pathlib import Path

PIPE = Path("/content/ba_pipeline/ba_pipeline")
LLM  = PIPE / "llm_outputs" / "runs"
SHAP = PIPE / "shap_outputs" / "local" / "A"
EVAL = PIPE / "results" / "eval_runs"

SEED_MISMATCH, JUDGE_STABILITY_N = 123, 5

rj = lambda p: json.loads(Path(p).read_text(encoding="utf-8"))
wj = lambda p,o: Path(p).write_text(json.dumps(o, ensure_ascii=False, indent=2), encoding="utf-8")
parse = lambda n: (int(n.split("_")[2]), int(n.split("_")[3][1:].split(".")[0]))  # llm_trx_<tx>_r<r>.json

# tx_id -> {shap_file, model_proba, top_features}
shap = {int(s["transaction_id"]): {"shap_file": fp.name, "model_proba": s.get("model_proba"), "top_features": s.get("top_features")}
        for fp in SHAP.glob("shap_trx_*.json")
        for s in [rj(fp)]}

def join(run, repeats=None):
    out = EVAL / run / "combined"; out.mkdir(parents=True, exist_ok=True)
    reps = None if repeats is None else set(repeats)
    n = 0
    for fp in sorted((LLM / run).glob("llm_trx_*_r*.json")):
        tx, r = parse(fp.name)
        if (reps is not None and r not in reps) or tx not in shap:
            continue
        llm = rj(fp)
        wj(out / f"join_trx_{tx}_r{r}.json", {
            "transaction_id": tx, "run_name": run, **shap[tx],
            "repeat": r, "llm_text": llm.get("llm_text"), "llm_file": fp.name
        })
        n += 1
    print(f"✅ {run}: wrote {n} -> {out}")

# 1) Benchmark r1, 2) Stability all repeats, 3) Other runs r1
join("run_001_benchmark", (1,))
join("run_002_stability")
for rn in ("run_003_ablation_weaker_model", "run_004_constraints_on"):
    join(rn, (1,))

# 4) Negative control: mismatched pairing (benchmark r1 text + SHAP from different tx)
src_run, out_name = "run_001_benchmark", "run_005_negative_control_mismatched_pairing"
out = EVAL / out_name / "combined"; out.mkdir(parents=True, exist_ok=True)

tx_to_fp = {parse(fp.name)[0]: fp for fp in sorted((LLM / src_run).glob("llm_trx_*_r1.json"))}
tx_ids   = sorted([tx for tx in tx_to_fp if tx in shap])
perm     = tx_ids[:]; random.Random(SEED_MISMATCH).shuffle(perm)
for i, tx in enumerate(tx_ids):
    if perm[i] == tx: perm[i], perm[(i+1) % len(tx_ids)] = perm[(i+1) % len(tx_ids)], perm[i]
mapping = dict(zip(tx_ids, perm))

for tx in tx_ids:
    fp, shap_tx = tx_to_fp[tx], mapping[tx]
    llm = rj(fp)
    wj(out / f"join_trx_{tx}_r1.json", {
        "transaction_id": tx, "run_name": out_name, "repeat": 1,
        "shap_transaction_id": shap_tx, **shap[shap_tx],
        "llm_text": llm.get("llm_text"), "llm_file": fp.name
    })
print(f"✅ {out_name}: wrote {len(tx_ids)} -> {out} (SEED={SEED_MISMATCH})")

# 5) Judge stability inputs:
judge_out = EVAL / "run_006_judge_stability" / "combined"; judge_out.mkdir(parents=True, exist_ok=True)
bench_dir = EVAL / "run_001_benchmark" / "combined"
for tx in tx_ids[:JUDGE_STABILITY_N]:
    fp = bench_dir / f"join_trx_{tx}_r1.json"
    if fp.exists(): wj(judge_out / fp.name, rj(fp))
print(f"✅ run_006_judge_stability: wrote {len(list(judge_out.glob('join_trx_*_r1.json')))} -> {judge_out}")

✅ run_001_benchmark: wrote 100 -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_001_benchmark/combined
✅ run_002_stability: wrote 50 -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_002_stability/combined
✅ run_003_ablation_weaker_model: wrote 100 -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_003_ablation_weaker_model/combined
✅ run_004_constraints_on: wrote 100 -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_004_constraints_on/combined
✅ run_005_negative_control_mismatched_pairing: wrote 100 -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_005_negative_control_mismatched_pairing/combined (SEED=123)
✅ run_006_judge_stability: wrote 5 -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_006_judge_stability/combined


# Define LLM-Judge-Prompt + Metrics

In [ ]:
import json, re
from pathlib import Path

ROOT = Path("/content/ba_pipeline")
PIPE = ROOT / "ba_pipeline"
EVAL_RUNS_DIR = PIPE / "results" / "eval_runs"

RUNS = [
    "run_001_benchmark",
    "run_002_stability",
    "run_003_ablation_weaker_model",
    "run_004_constraints_on",
    "run_005_negative_control_mismatched_pairing",
    "run_006_judge_stability",
]

PROMPT_TEMPLATE = """You are an evaluator in the domain of Anti-Money-Laundering.

INPUT:
1. SHAP_JSON = JSON array of {feature, value, shap_value, direction}
2. LLM_TEXT = LLM generated explanation of why a transaction is flagged as suspicious or not.
3. FACT_SHEET = Only defines feature-name semantics

TASK:
Score the LLM_TEXT for:
(1) Direction alignment,
(2) Driver coverage,
(3) No unsupported claims,
(4) Composition,
(5) Context utility.

FACT_SHEET:
    "from_bank": "sending bank ID",
    "to_bank": "receiving bank ID",
    "amount_received": "amount received (receiving currency)",
    "amount_paid": "amount paid (payment currency)",
    "hour": "hour of day (0–23)",
    "dayofweek": "day of week (0=Mon..6=Sun)",
    "amount_ratio": "amount_paid / amount_received",
    "amount_diff": "|amount_paid − amount_received|",
    "payment_format_": "payment method indicator (1=yes, 0=no)",
    "receiving_currency_": "receiving currency indicator (1=this, 0=other)",
    "payment_currency_": "payment currency indicator (1=this, 0=other)"

ONE-HOT SEMANTICS:
One-hot currency/payment-format features are those whose feature names contain "currency_" or "payment_format_".
- value = 1 supports text meaning "is X"
- value = 0 supports text meaning "is not X" / "non-X"

METRICS:
A) Direction alignment (0/1)
Check that every driver direction (increase/decrease) stated in LLM_TEXT about a provided SHAP feature matches that feature’s direction label in SHAP_JSON; any mismatch fails.
Score:
1 = Pass: No such driver-linked direction statement in LLM_TEXT contradicts the corresponding direction in SHAP_JSON.
0 = Fail: At least one such driver-linked direction statement in LLM_TEXT contradicts the corresponding direction in SHAP_JSON.

B) Driver coverage (0/1/2)
Computation:
1. Take the list of provided SHAP-drivers from SHAP_JSON and look at the absolute size of each driver's shap_value (ignore the sign). Add these up to get the total shap_value available in the provided list.
2. Read the narrative and mark which of those drivers it actually mentions (i.e., it clearly refers to that feature). For just the mentioned drivers, again take the absolute shap_value and add them up.
3. Divide the "total mentioned shap_values" by the "total shap_values" to get the driver_coverage.
4. Convert that fraction into the score.
Score:
2 = driver_coverage ≥ 0.80
1 = 0.50 ≤ driver_coverage < 0.80
0 = driver_coverage < 0.50

C) no_unsupported_claims (0/1)
Check whether the narrative makes any unsupported factual statements: a claim counts as unsupported if it states a fact that cannot be linked to a provided driver.
The score is 1 only if there are zero such claims; otherwise it is 0.
Score:
1 = No UNSUPPORTED CLAIM exists.
0 = At least one UNSUPPORTED CLAIM exists.

D) composition (0–4)
This is a rubric score for the form of the writing only: how clear, well-structured, and non-repetitive the explanation is.
It explicitly ignores factual correctness and scores only readability/organization.
Score:
4 = very clear; well-structured; minimal repetition; grammar never blocks understanding
3 = clear overall; minor wording/structure issues
2 = readable but clunky; noticeable repetition; weak cohesion
1 = hard to follow; poorly organized; frequent confusing phrasing
0 = incoherent/unreadable

E) context_utility (0–4)
This is a rubric score for how useful the text would be to an AML investigator given only the provided drivers (e.g., does it clearly explain the main risk signal and why it matters, without adding facts).
4 = concrete, clear rationale tied to factors; easy to act on
3 = generally helpful but somewhat generic
2 = limited; mostly restates factors with little interpretation
1 = vague/template-like; little interpretive value
0 = misleading/contradictory/harmful

OUTPUT:
Return VALID JSON ONLY with exactly these keys:
{
  "direction_alignment": 0|1,
  "driver_coverage": 0|1|2,
  "no_unsupported_claims": 0|1,
  "composition": 0|1|2|3|4,
  "context_utility": 0|1|2|3|4,
  "note": "max 1 sentence; mention the single biggest issue or strongest point"
}

SHAP_JSON:
<<<SHAP_JSON>>>

LLM_TEXT:
<<<LLM_TEXT>>>"""

def write_txt(fp: Path, top_features, llm_text: str):
    txt = PROMPT_TEMPLATE.replace("<<<SHAP_JSON>>>", json.dumps(top_features, ensure_ascii=False))
    txt = txt.replace("<<<LLM_TEXT>>>", llm_text or "")
    fp.write_text(txt, encoding="utf-8")

JUDGE_STABILITY_REPEATS = 10
pat = re.compile(r"join_trx_(\d+)_r(\d+)\.json$")

total = 0
for run in RUNS:
    src_dir = EVAL_RUNS_DIR / run / "combined"
    out_dir = EVAL_RUNS_DIR / run / "judge_inputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    for fp in sorted(src_dir.glob("join_trx_*_r*.json")):
        j = json.loads(fp.read_text(encoding="utf-8"))
        assert j.get("top_features") is not None, f"Missing top_features in {fp}"

        m = pat.match(fp.name)
        tx_id = int(m.group(1)) if m else int(j["transaction_id"])
        r_src = int(m.group(2)) if m else int(j["repeat"])

        if run != "run_006_judge_stability":
            write_txt(out_dir / f"judge_trx_{tx_id}_r{r_src}.txt", j["top_features"], j.get("llm_text"))
            total += 1
        else:
            for r in range(1, JUDGE_STABILITY_REPEATS + 1):
                write_txt(out_dir / f"judge_trx_{tx_id}_r{r}.txt", j["top_features"], j.get("llm_text"))
                total += 1

print(f"✅ wrote {total} judge input .txt files")

✅ wrote 500 judge input .txt files


# API calls

In [ ]:
import os, json, time, re
from pathlib import Path
from openai import OpenAI
from tqdm import tqdm

# --- API key
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
    except Exception:
        pass
if not os.getenv("OPENAI_API_KEY") and "OPENAI_API_KEY" in globals():
    os.environ["OPENAI_API_KEY"] = globals()["OPENAI_API_KEY"]
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

ROOT = Path("/content/ba_pipeline")
PIPE = ROOT / "ba_pipeline"
EVAL_RUNS_DIR = PIPE / "results" / "eval_runs"

RUNS = [
    "run_001_benchmark",
    "run_002_stability",
    "run_003_ablation_weaker_model",
    "run_004_constraints_on",
    "run_005_negative_control_mismatched_pairing",
    "run_006_judge_stability",
]

MODEL = "gpt-5.2"
client = OpenAI()

MAX_PER_RUN = None  # None for all files
SLEEP_S = 0.2
OVERWRITE = True   # False = skip existing outputs

json_obj_re = re.compile(r"\{.*\}", re.DOTALL)

for run in RUNS:
    inp_dir = EVAL_RUNS_DIR / run / "judge_inputs"
    out_dir = EVAL_RUNS_DIR / run / "judge_out"
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(inp_dir.glob("judge_trx_*_r*.txt"))
    if MAX_PER_RUN is not None:
        files = files[:MAX_PER_RUN]

    for fp in tqdm(files, desc=run):
        out_fp = out_dir / fp.with_suffix(".json").name.replace("judge_trx_", "judge_out_")
        if out_fp.exists() and not OVERWRITE:
            continue

        try:
            prompt = fp.read_text(encoding="utf-8")

            resp = client.responses.create(
                model=MODEL,
                input=prompt,
            )
            txt = (resp.output_text or "").strip()

            m = json_obj_re.search(txt)
            if not m:
                raise ValueError(f"Judge did not return JSON. First 300 chars:\n{txt[:300]}")
            obj = json.loads(m.group(0))

            out_fp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

        except Exception as e:
            print("ERROR", run, fp.name, "->", repr(e))

        time.sleep(SLEEP_S)

run_006_judge_stability: 100%|██████████| 50/50 [01:54<00:00,  2.28s/it]


In [ ]:
import json, re
from pathlib import Path
from tqdm import tqdm

ROOT = Path("/content/ba_pipeline")
PIPE = ROOT / "ba_pipeline"
EVAL_RUNS_DIR = PIPE / "results" / "eval_runs"

RUNS = [
    "run_001_benchmark",
    "run_002_stability",
    "run_003_ablation_weaker_model",
    "run_004_constraints_on",
    "run_005_negative_control_mismatched_pairing",
    "run_006_judge_stability",
]

# matches judge_out_<tx>_r<r>.json
pat = re.compile(r"judge_out_(\d+)_r(\d+)\.json$")

def word_count_whitespace(s: str) -> int:
    return len((s or "").strip().split())

WORD_LIMIT = 200

for run in RUNS:
    combined_dir = EVAL_RUNS_DIR / run / "combined"
    judge_dir    = EVAL_RUNS_DIR / run / "judge_out"
    final_dir    = EVAL_RUNS_DIR / run / "final_results"
    final_dir.mkdir(parents=True, exist_ok=True)

    judge_files = sorted(judge_dir.glob("judge_out_*_r*.json"))
    if not judge_files:
        print(f"[{run}] no judge_out files found -> skip")
        continue

    miss = 0
    for jfp in tqdm(judge_files, desc=f"{run} | finalize"):
        m = pat.match(jfp.name)
        if not m:
            continue
        tx_id, r = int(m.group(1)), int(m.group(2))

        # join lookup (run_006 only has r1 join files; reuse r1 for judge-only repeats)
        join_fp = combined_dir / f"join_trx_{tx_id}_r{r}.json"
        if not join_fp.exists() and run == "run_006_judge_stability":
            join_fp = combined_dir / f"join_trx_{tx_id}_r1.json"

        if not join_fp.exists():
            miss += 1
            print(f"MISS join: {run} {join_fp.name}")
            continue

        judge_obj = json.loads(jfp.read_text(encoding="utf-8"))
        join_obj  = json.loads(join_fp.read_text(encoding="utf-8"))

        wc = word_count_whitespace(join_obj.get("llm_text"))

        out = dict(judge_obj)
        out.update({
            "transaction_id": tx_id,
            "repeat": r,
            "run_name": run,
            "word_count": wc,
            "word_limit": WORD_LIMIT,
            "word_limit_pass": 1 if wc <= WORD_LIMIT else 0,  # 0/1
        })

        out_fp = final_dir / jfp.name.replace("judge_out_", "final_")
        out_fp.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[{run}] ✅ wrote -> {final_dir} (miss={miss})")

run_001_benchmark | finalize: 100%|██████████| 100/100 [00:00<00:00, 2583.37it/s]


[run_001_benchmark] ✅ wrote -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_001_benchmark/final_results (miss=0)


run_002_stability | finalize: 100%|██████████| 50/50 [00:00<00:00, 2773.64it/s]


[run_002_stability] ✅ wrote -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_002_stability/final_results (miss=0)


run_003_ablation_weaker_model | finalize: 100%|██████████| 100/100 [00:00<00:00, 1837.83it/s]


...

[run_004_constraints_on] ✅ wrote -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_004_constraints_on/final_results (miss=0)


run_005_negative_control_mismatched_pairing | finalize: 100%|██████████| 100/100 [00:00<00:00, 2952.86it/s]


[run_005_negative_control_mismatched_pairing] ✅ wrote -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_005_negative_control_mismatched_pairing/final_results (miss=0)


run_006_judge_stability | finalize: 100%|██████████| 50/50 [00:00<00:00, 2543.70it/s]

[run_006_judge_stability] ✅ wrote -> /content/ba_pipeline/ba_pipeline/results/eval_runs/run_006_judge_stability/final_results (miss=0)


# Git Push (First)

In [ ]:
import subprocess
from pathlib import Path
from urllib.parse import quote
from google.colab import userdata

REPO_URL  = "https://github.com/ZeusGav/evaluating-llm-explanations-for-aml.git"
REPO_DIR  = Path("/content/ba_pipeline")
BRANCH    = "main"
MSG       = "Sync pipeline outputs"

def sh(cmd, cwd=REPO_DIR):
    subprocess.run(cmd, cwd=cwd, check=True)

assert (REPO_DIR / ".git").exists(), f"Not a git repo: {REPO_DIR}"

# auth remote
token = userdata.get("GITHUB_TOKEN")
assert token, "Missing GITHUB_TOKEN in Colab secrets"
authed = REPO_URL.replace("https://", f"https://ZeusGav:{quote(token, safe='')}@")
sh(["git", "remote", "set-url", "origin", authed])
sh(["git", "config", "user.name", "colab"])
sh(["git", "config", "user.email", "colab@local"])

# stage + commit + push
sh(["git", "add", "-A"])

status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()

if not status:
    print("No changes to commit.")
else:
    sh(["git", "commit", "-m", MSG])
    sh(["git", "push", "-u", "origin", BRANCH])
    print("Pushed.")

Pushed.


# Pipeline 2.1 Evaluation


In [ ]:
from pathlib import Path
import shutil, subprocess

REPO_URL = "https://github.com/ZeusGav/evaluating-llm-explanations-for-aml.git"
REPO_DIR = Path("/content/ba_pipeline")
PIPE     = REPO_DIR / "ba_pipeline"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

CompletedProcess(args=['git', 'clone', 'https://github.com/ZeusGav/evaluating-llm-explanations-for-aml.git', '/content/ba_pipeline'], returncode=0)

In [ ]:
# paths + helpers

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/ba_pipeline/ba_pipeline")
EVAL, FIGS, OUT = ROOT/"results/eval_runs", ROOT/"figures", ROOT/"results/eval_summary"
FIGS.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

RUNS_ALL = ["run_001_benchmark","run_002_stability","run_003_ablation_weaker_model",
            "run_004_constraints_on","run_005_negative_control_mismatched_pairing","run_006_judge_stability"]
RUNS_STABILITY = ["run_002_stability","run_006_judge_stability"]
RUNS_MAIN = [r for r in RUNS_ALL if r not in RUNS_STABILITY]

MET_ORDER = [("direction_alignment","M-01"),("driver_coverage","M-02"),("no_unsupported_claims","M-03"),
             ("composition","M-04"),("context_utility","M-05"),("word_count","M-06"),("word_limit_pass","M-06_pass")]

SEED = 42
read_json = lambda fp: json.loads(Path(fp).read_text(encoding="utf-8"))

def bootstrap_ci_mean(x, n_boot=3000, alpha=0.05, seed=SEED):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if x.size == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    boots = rng.choice(x, size=(n_boot, x.size), replace=True).mean(axis=1)
    return (float(x.mean()), float(np.quantile(boots, alpha/2)), float(np.quantile(boots, 1-alpha/2)))

def bootstrap_ci_diff(a, b, n_boot=3000, alpha=0.05, seed=SEED):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = ~np.isnan(a) & ~np.isnan(b)
    a, b = a[m], b[m]
    if a.size == 0:
        return (np.nan, np.nan, np.nan)

    d = a - b
    rng = np.random.default_rng(seed)
    boots = rng.choice(d, size=(n_boot, d.size), replace=True).mean(axis=1)
    return (float(d.mean()),
            float(np.quantile(boots, alpha/2)),
            float(np.quantile(boots, 1 - alpha/2)))

In [ ]:
# Load data

rows=[]
for run in RUNS_ALL:
    d = EVAL/run/"final_results"
    if not d.exists(): print(f"[WARN] missing: {run}"); continue
    for fp in d.glob("final_*.json"):
        j = read_json(fp)
        rows.append({
            "run": run, "transaction_id": int(j["transaction_id"]), "repeat": int(j["repeat"]),
            "direction_alignment": j.get("direction_alignment"),
            "driver_coverage": j.get("driver_coverage"),
            "no_unsupported_claims": j.get("no_unsupported_claims"),
            "composition": j.get("composition"),
            "context_utility": j.get("context_utility"),
            "word_count": j.get("word_count"),
            "word_limit_pass": j.get("word_limit_pass"),
        })

df = pd.DataFrame(rows)
assert not df.empty, "No final_results found."
for c in df.columns.difference(["run"]): df[c] = pd.to_numeric(df[c], errors="coerce")
df.to_csv(OUT/"eval_tidy.csv", index=False)
print(f"✅ rows={len(df)} | saved={OUT/'eval_tidy.csv'}")

✅ rows=500 | saved=/content/ba_pipeline/ba_pipeline/results/eval_summary/eval_tidy.csv


In [ ]:
# AGGREGATE tx-level and then per-run + bootstrap CIs (main runs only)

tx = (df[df["run"].isin(RUNS_MAIN)]
      .groupby(["run","transaction_id"], as_index=False)
      .agg(n_repeats=("repeat","nunique"),
           direction_alignment=("direction_alignment","mean"),
           driver_coverage=("driver_coverage","mean"),
           no_unsupported_claims=("no_unsupported_claims","mean"),
           composition=("composition","mean"),
           context_utility=("context_utility","mean"),
           word_count=("word_count","mean"),
           word_limit_pass=("word_limit_pass","mean")))
tx.to_csv(OUT/"summary_by_run_tx_level.csv", index=False)

rows=[]
for run,g in tx.groupby("run"):
    r={"run":run,"n_tx":len(g)}
    for m in ["direction_alignment","driver_coverage","no_unsupported_claims","composition","context_utility","word_count","word_limit_pass"]:
        mean,lo,hi = bootstrap_ci_mean(g[m].values)
        r[f"{m}_mean"], r[f"{m}_ci_lo"], r[f"{m}_ci_hi"] = mean,lo,hi
    rows.append(r)

summary = pd.DataFrame(rows).sort_values("run")
summary.to_csv(OUT/"summary_by_run.csv", index=False)
print(f"✅ saved: {OUT/'summary_by_run.csv'} + {OUT/'summary_by_run_tx_level.csv'}")
summary

✅ saved: /content/ba_pipeline/ba_pipeline/results/eval_summary/summary_by_run.csv + /content/ba_pipeline/ba_pipeline/results/eval_summary/summary_by_run_tx_level.csv


,run,n_tx,direction_alignment_mean,direction_alignment_ci_lo,direction_alignment_ci_hi,driver_coverage_mean,driver_coverage_ci_lo,driver_coverage_ci_hi,no_unsupported_claims_mean,no_unsupported_claims_ci_lo,...,composition_ci_hi,context_utility_mean,context_utility_ci_lo,context_utility_ci_hi,word_count_mean,word_count_ci_lo,word_count_ci_hi,word_limit_pass_mean,word_limit_pass_ci_lo,word_limit_pass_ci_hi
0,run_001_benchmark,100,1.00,1.00,1.00,1.96,1.91,1.99,0.67,0.57975,...,4.0,3.23,3.15,3.31,171.29,167.26975,175.39075,0.91,0.85,0.96
1,run_003_ablation_weaker_model,100,0.97,0.93,1.00,1.91,1.85,1.96,0.41,0.32000,...,4.0,3.13,3.07,3.20,181.70,174.65000,188.72050,0.72,0.63,0.80
2,run_004_constraints_on,100,0.99,0.97,1.00,1.95,1.90,1.99,0.55,0.45975,...,4.0,3.34,3.25,3.43,117.90,115.24975,120.57025,1.00,1.00,1.00
3,run_005_negative_control_mismatched_pairing,100,0.44,0.34,0.54,1.23,1.13,1.33,0.01,0.00000,...,3.9,2.72,2.61,2.81,171.29,167.26975,175.39075,0.91,0.85,0.96


In [ ]:
# STABILITY (variance): run_002 / run_006

STABILITY_RUNS = ["run_002_stability", "run_006_judge_stability"]
METRICS = ["direction_alignment", "driver_coverage", "no_unsupported_claims", "composition", "context_utility"]

stab = df[df["run"].isin(STABILITY_RUNS)]

var_if_rep = lambda x: np.var(x, ddof=1) if len(x) >= 2 else np.nan

stab_tx = (stab.groupby(["run","transaction_id"], as_index=False)
             .agg(n_repeats=("repeat","nunique"),
                  **{f"var_{m}": (m, var_if_rep) for m in METRICS}))

stab_tx.to_csv(OUT / "stability_by_tx.csv", index=False)

rows = []
for run, g in stab_tx.groupby("run"):
    row = {"run": run, "n_tx": g["transaction_id"].nunique()}
    for m in METRICS:
        mean, lo, hi = bootstrap_ci_mean(g[f"var_{m}"].dropna().values)
        row.update({f"var_{m}_mean": mean, f"var_{m}_ci_lo": lo, f"var_{m}_ci_hi": hi})
    rows.append(row)

stab_summary = pd.DataFrame(rows).sort_values("run")
stab_summary.to_csv(OUT / "stability_by_run.csv", index=False)
print("✅ saved:", OUT / "stability_by_tx.csv", "and", OUT / "stability_by_run.csv")

stab_summary

✅ saved: /content/ba_pipeline/ba_pipeline/results/eval_summary/stability_by_tx.csv and /content/ba_pipeline/ba_pipeline/results/eval_summary/stability_by_run.csv


,run,n_tx,var_direction_alignment_mean,var_direction_alignment_ci_lo,var_direction_alignment_ci_hi,var_driver_coverage_mean,var_driver_coverage_ci_lo,var_driver_coverage_ci_hi,var_no_unsupported_claims_mean,var_no_unsupported_claims_ci_lo,var_no_unsupported_claims_ci_hi,var_composition_mean,var_composition_ci_lo,var_composition_ci_hi,var_context_utility_mean,var_context_utility_ci_lo,var_context_utility_ci_hi,var_word_limit_pass_mean,var_word_limit_pass_ci_lo,var_word_limit_pass_ci_hi
0,run_002_stability,5,0.0,0.0,0.0,0.060000,0.02,0.100000,0.206667,0.177778,0.242222,0.0,0.0,0.0,0.184444,0.075556,0.275556,0.086667,0.02,0.1605
1,run_006_judge_stability,5,0.0,0.0,0.0,0.075556,0.02,0.131111,0.128889,0.055556,0.213333,0.0,0.0,0.0,0.106667,0.000000,0.213333,0.000000,0.00,0.0000


In [ ]:
# Comparison: Benchmark (R1) vs other runs (tx-level) + bootstrap CI

bench = tx[tx["run"] == "run_001_benchmark"].set_index("transaction_id")

delta_rows = []
for run in RUNS_ALL:
    if run == "run_001_benchmark":
        continue
    g = tx[tx["run"] == run].set_index("transaction_id")
    common = bench.index.intersection(g.index)
    if len(common) == 0:
        continue

    row = {"run": run, "n_tx_common": int(len(common))}
    for m in ["direction_alignment","no_unsupported_claims","driver_coverage","composition","context_utility","word_count"]:
        d_mean, d_lo, d_hi = bootstrap_ci_diff(g.loc[common, m].values, bench.loc[common, m].values)
        row[f"delta_{m}"] = d_mean
        row[f"delta_{m}_ci_lo"] = d_lo
        row[f"delta_{m}_ci_hi"] = d_hi
    delta_rows.append(row)

deltas = pd.DataFrame(delta_rows).sort_values("run")
deltas.to_csv(OUT / "deltas_vs_benchmark.csv", index=False)
print("✅ saved:", OUT / "deltas_vs_benchmark.csv")
deltas

✅ saved: /content/ba_pipeline/ba_pipeline/results/eval_summary/deltas_vs_benchmark.csv


,run,n_tx_common,delta_direction_alignment,delta_direction_alignment_ci_lo,delta_direction_alignment_ci_hi,delta_no_unsupported_claims,delta_no_unsupported_claims_ci_lo,delta_no_unsupported_claims_ci_hi,delta_driver_coverage,delta_driver_coverage_ci_lo,delta_driver_coverage_ci_hi,delta_composition,delta_composition_ci_lo,delta_composition_ci_hi,delta_context_utility,delta_context_utility_ci_lo,delta_context_utility_ci_hi,delta_word_count,delta_word_count_ci_lo,delta_word_count_ci_hi
0,run_003_ablation_weaker_model,100,-0.03,-0.07,0.00,-0.26,-0.39,-0.13,-0.05,-0.11,0.01,0.00,0.00,0.0,-0.10,-0.20,0.00,10.41,2.15975,18.7705
1,run_004_constraints_on,100,-0.01,-0.03,0.00,-0.12,-0.24,0.01,-0.01,-0.06,0.03,0.00,0.00,0.0,0.11,0.00,0.23,-53.39,-57.96100,-48.7700
2,run_005_negative_control_mismatched_pairing,100,-0.56,-0.66,-0.46,-0.66,-0.75,-0.57,-0.73,-0.84,-0.63,-0.17,-0.25,-0.1,-0.51,-0.64,-0.38,0.00,0.00000,0.0000


# Plots

In [ ]:
# Plotting

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/content/ba_pipeline/ba_pipeline")
OUT, FIGS = ROOT/"results/eval_summary", ROOT/"figures"
FIGS.mkdir(parents=True, exist_ok=True)

SUMMARY_FP, STABILITY_FP = OUT/"summary_by_run.csv", OUT/"stability_by_run.csv"
TX_LEVEL_FP = OUT/"eval_tidy.csv"

RUN_LABELS = {
    "run_001_benchmark": "Run 1 — Benchmark",
    "run_003_ablation_weaker_model": "Run 3 — Weaker model",
    "run_004_constraints_on": "Run 4 — Constraints on",
    "run_005_negative_control_mismatched_pairing": "Run 5 — Negative control",
    "run_002_stability": "Run 2 — Stability (end-to-end)",
    "run_006_judge_stability": "Run 6 — Judge stability",
}
MAIN_RUNS = ["run_001_benchmark","run_003_ablation_weaker_model","run_004_constraints_on","run_005_negative_control_mismatched_pairing"]
STAB_RUNS = ["run_002_stability","run_006_judge_stability"]

def need(df, cols, name):
    miss = [c for c in cols if c not in df.columns]
    if miss: raise ValueError(f"{name} missing columns: {miss}")

def bar_ci(ax, labels, mean, lo, hi, title, ylim=None, rot=25):
    x = np.arange(len(labels))
    m, lo, hi = map(lambda v: np.asarray(v, float), (mean, lo, hi))
    ax.bar(x, m, yerr=np.vstack([m-lo, hi-m]), capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=rot, ha="right")
    ax.set_title(title); ax.grid(axis="y", alpha=0.3)
    if ylim: ax.set_ylim(*ylim)

# Load + prepare
summary = pd.read_csv(SUMMARY_FP)
stab    = pd.read_csv(STABILITY_FP)
need(summary, ["run"], "summary"); need(stab, ["run"], "stability")

summary["run_label"] = summary["run"].map(RUN_LABELS).fillna(summary["run"])
stab["run_label"]    = stab["run"].map(RUN_LABELS).fillna(stab["run"])

summary_main = summary[summary["run"].isin(MAIN_RUNS)].copy()
summary_main["run"] = pd.Categorical(summary_main["run"], categories=MAIN_RUNS, ordered=True)
summary_main = summary_main.sort_values("run")
labels = summary_main["run_label"].tolist()

# Plot A: pass rates (M-01, M-03, M-06_pass)
need(summary_main, [
    "direction_alignment_mean","direction_alignment_ci_lo","direction_alignment_ci_hi",
    "no_unsupported_claims_mean","no_unsupported_claims_ci_lo","no_unsupported_claims_ci_hi",
], "summary_main")

fig = plt.figure(figsize=(11, 3.6))
gs = fig.add_gridspec(1, 3, wspace=0.35)

ax = fig.add_subplot(gs[0,0])
bar_ci(ax, labels,
       summary_main["direction_alignment_mean"], summary_main["direction_alignment_ci_lo"], summary_main["direction_alignment_ci_hi"],
       "M-01 Direction alignment", ylim=(0,1.05))

ax = fig.add_subplot(gs[0,1])
bar_ci(ax, labels,
       summary_main["no_unsupported_claims_mean"], summary_main["no_unsupported_claims_ci_lo"], summary_main["no_unsupported_claims_ci_hi"],
       "M-03 No unsupported claims", ylim=(0,1.05))

ax = fig.add_subplot(gs[0,2])
if all(c in summary_main.columns for c in ["word_limit_pass_mean","word_limit_pass_ci_lo","word_limit_pass_ci_hi"]):
    bar_ci(ax, labels,
           summary_main["word_limit_pass_mean"], summary_main["word_limit_pass_ci_lo"], summary_main["word_limit_pass_ci_hi"],
           "M-06 Pass rate (≤ 200 words)", ylim=(0,1.05))
else:
    ax.axis("off"); ax.text(0.5,0.5,"M-06_pass not in summary CSV", ha="center", va="center")

fig.tight_layout()
fig.savefig(FIGS/"plot_A_M01_M03_M06.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# Plot B: tx-level word count boxplot
if TX_LEVEL_FP.exists():
    tx = pd.read_csv(TX_LEVEL_FP)
    need(tx, ["run","word_count"], "eval_tidy")
    tx = tx[tx["run"].isin(MAIN_RUNS)].copy()
    tx["run"] = pd.Categorical(tx["run"], categories=MAIN_RUNS, ordered=True)
    tx = tx.sort_values("run")

    fig, ax = plt.subplots(figsize=(8.8, 3.4))
    groups = [tx.loc[tx["run"]==r, "word_count"].dropna().to_numpy(float) for r in MAIN_RUNS]
    ax.boxplot(groups, labels=[RUN_LABELS[r] for r in MAIN_RUNS], showfliers=False)
    ax.axhline(200, linestyle="--", linewidth=1)
    ax.set_title("M-06 Word count by run (transaction-level)")
    ax.set_ylabel("Word count"); ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGS/"plot_B_word_count_box.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

# Plot C: rubrics (M-02, M-04, M-05)
need(summary_main, [
    "driver_coverage_mean","driver_coverage_ci_lo","driver_coverage_ci_hi",
    "composition_mean","composition_ci_lo","composition_ci_hi",
    "context_utility_mean","context_utility_ci_lo","context_utility_ci_hi",
], "summary_main")

fig = plt.figure(figsize=(11, 3.6))
gs = fig.add_gridspec(1, 3, wspace=0.35)

ax = fig.add_subplot(gs[0,0])
bar_ci(ax, labels,
       summary_main["driver_coverage_mean"], summary_main["driver_coverage_ci_lo"], summary_main["driver_coverage_ci_hi"],
       "M-02 Driver coverage (mean score)", ylim=(0,2.05))

ax = fig.add_subplot(gs[0,1])
bar_ci(ax, labels,
       summary_main["composition_mean"], summary_main["composition_ci_lo"], summary_main["composition_ci_hi"],
       "M-04 Composition (mean score)", ylim=(0,4.05))

ax = fig.add_subplot(gs[0,2])
bar_ci(ax, labels,
       summary_main["context_utility_mean"], summary_main["context_utility_ci_lo"], summary_main["context_utility_ci_hi"],
       "M-05 Context utility (mean score)", ylim=(0,4.05))

fig.tight_layout()
fig.savefig(FIGS/"plot_C_rubrics_M02_M04_M05.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# Plot D: stability variances (run_002 vs run_006)
stab2 = stab[stab["run"].isin(STAB_RUNS)].copy()
stab2["run"] = pd.Categorical(stab2["run"], categories=STAB_RUNS, ordered=True)
stab2 = stab2.sort_values("run")

var_metrics = [
    ("var_direction_alignment", "M-01"),
    ("var_driver_coverage", "M-02"),
    ("var_no_unsupported_claims", "M-03"),
    ("var_composition", "M-04"),
    ("var_context_utility", "M-05"),
]
need(stab2, ["run"] + [f"{k}_{s}" for k,_ in var_metrics for s in ("mean","ci_lo","ci_hi")], "stability_by_run")

xlab = [m for _, m in var_metrics]
x = np.arange(len(xlab)); w = 0.38

fig, ax = plt.subplots(figsize=(9.8, 3.8))
for i, run in enumerate(STAB_RUNS):
    row = stab2.loc[stab2["run"]==run].iloc[0]
    mean = np.array([row[f"{k}_mean"] for k,_ in var_metrics], float)
    lo   = np.array([row[f"{k}_ci_lo"] for k,_ in var_metrics], float)
    hi   = np.array([row[f"{k}_ci_hi"] for k,_ in var_metrics], float)
    ax.bar(x + (i-0.5)*w, mean, w, yerr=np.vstack([mean-lo, hi-mean]), capsize=4, label=RUN_LABELS[run])

ax.set_xticks(x); ax.set_xticklabels(xlab)
ax.set_title("Stability: mean per-transaction variance (ddof=1) with 95% CIs")
ax.set_ylabel("Variance"); ax.grid(axis="y", alpha=0.3); ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(FIGS/"plot_D_stability_variance.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print("✅ Saved figures to:", FIGS)

/tmp/ipython-input-4230842043.py:85: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipython-input-4230842043.py:101: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(groups, labels=[RUN_LABELS[r] for r in MAIN_RUNS], showfliers=False)
/tmp/ipython-input-4230842043.py:137: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


✅ Saved figures to: /content/ba_pipeline/ba_pipeline/figures


# Git Push Second

In [ ]:
import subprocess
from pathlib import Path
from urllib.parse import quote
from google.colab import userdata

REPO_URL  = "https://github.com/ZeusGav/evaluating-llm-explanations-for-aml.git"
REPO_DIR  = Path("/content/ba_pipeline")
BRANCH    = "main"
MSG       = "Sync pipeline outputs"

def sh(cmd, cwd=REPO_DIR):
    subprocess.run(cmd, cwd=cwd, check=True)

assert (REPO_DIR / ".git").exists(), f"Not a git repo: {REPO_DIR}"

# auth remote
token = userdata.get("GITHUB_TOKEN")
assert token, "Missing GITHUB_TOKEN in Colab secrets"
authed = REPO_URL.replace("https://", f"https://ZeusGav:{quote(token, safe='')}@")
sh(["git", "remote", "set-url", "origin", authed])
sh(["git", "config", "user.name", "colab"])
sh(["git", "config", "user.email", "colab@local"])

# stage + commit + push
sh(["git", "add", "-A"])

status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()

if not status:
    print("No changes to commit.")
else:
    sh(["git", "commit", "-m", MSG])
    sh(["git", "push", "-u", "origin", BRANCH])
    print("Pushed.")

Pushed.
